In [1]:
import pandas as pd

In [2]:
# import data
schedules = pd.read_csv('Data/schedules.csv')
player_stats = pd.read_csv('Data/player_stats.csv')
team_stats = pd.read_csv('Data/team_stats.csv')

In [3]:
schedules.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,1999_01_MIN_ATL,1999,REG,1,1999-09-12,Sunday,NaN,MIN,17,ATL,...,NaN,00-0003761,00-0002876,Randall Cunningham,Chris Chandler,Dennis Green,Dan Reeves,Gerry Austin,ATL00,Georgia Dome
1,1999_01_KC_CHI,1999,REG,1,1999-09-12,Sunday,NaN,KC,17,CHI,...,12.0,00-0006300,00-0010560,Elvis Grbac,Shane Matthews,Gunther Cunningham,Dick Jauron,Phil Luckett,CHI98,Soldier Field
2,1999_01_PIT_CLE,1999,REG,1,1999-09-12,Sunday,NaN,PIT,43,CLE,...,12.0,00-0015700,00-0004230,Kordell Stewart,Ty Detmer,Bill Cowher,Chris Palmer,Bob McElwee,CLE00,Cleveland Browns Stadium
3,1999_01_OAK_GB,1999,REG,1,1999-09-12,Sunday,NaN,OAK,24,GB,...,10.0,00-0005741,00-0005106,Rich Gannon,Brett Favre,Jon Gruden,Ray Rhodes,Tony Corrente,GNB00,Lambeau Field
4,1999_01_BUF_IND,1999,REG,1,1999-09-12,Sunday,NaN,BUF,14,IND,...,NaN,00-0005363,00-0010346,Doug Flutie,Peyton Manning,Wade Phillips,Jim Mora,Ron Blum,IND99,RCA Dome


In [6]:
# create dataframe of team records by year, have to account for them being both home and away
import numpy as np

# 1. Determine the game outcomes from the perspective of home and away
schedules["Home_Win"] = np.where(schedules["home_score"] > schedules["away_score"], 1, 0)
schedules["Home_Loss"] = np.where(schedules["home_score"] < schedules["away_score"], 1, 0)
schedules["Away_Win"] = schedules["Home_Loss"]  # If home lost, away won
schedules["Away_Loss"] = schedules["Home_Win"]  # If home won, away lost

# Handle ties (if applicable to your sport)
schedules["Tie"] = np.where(schedules["home_score"] == schedules["away_score"], 1, 0)

# 2. Aggregate Home Records
home_rec = (
    schedules.groupby(["season", "home_team"])[["Home_Win", "Home_Loss", "Tie"]]
    .sum()
    .reset_index()
)
home_rec.rename(
    columns={
        "home_team": "Team",
        "Home_Win": "Wins",
        "Home_Loss": "Losses",
        "Tie": "Ties",
    },
    inplace=True,
)

# 3. Aggregate Away Records
away_rec = (
    schedules.groupby(["season", "away_team"])[["Away_Win", "Away_Loss", "Tie"]]
    .sum()
    .reset_index()
)
away_rec.rename(
    columns={
        "away_team": "Team",
        "Away_Win": "Wins",
        "Away_Loss": "Losses",
        "Tie": "Ties",
    },
    inplace=True,
)

# 4. Combine and Sum
# Concatenate them vertically, then group by Season and Team to sum them up
season_records = (
    pd.concat([home_rec, away_rec])
    .groupby(["season", "Team"])[["Wins", "Losses", "Ties"]]
    .sum()
    .reset_index()
)

# Optional: Add a Win Percentage column for cleaner analysis
total_games = season_records["Wins"] + season_records["Losses"] + season_records["Ties"]
season_records["Win_Pct"] = (
    (season_records["Wins"] + 0.5 * season_records["Ties"]) / total_games
).round(3)

season_records.head()

,season,Team,Wins,Losses,Ties,Win_Pct
0,1999,ARI,6,10,0,0.375
1,1999,ATL,5,11,0,0.312
2,1999,BAL,8,8,0,0.500
3,1999,BUF,11,6,0,0.647
4,1999,CAR,8,8,0,0.500


In [7]:
# save records to CSV
season_records.to_csv("Data/team_records.csv", index=False)